# 英国道路碰撞严重程度预测

本 Notebook 是项目的轻量、可执行入口。它直接读取受版本控制的结果快照和报告图，
不再复制训练实现。流程分为四个阶段：原始数据验证、描述性分析、五模型时间外验证，
以及独立测试年度评估和补充诊断。

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

START = Path.cwd().resolve()
ROOT = next(
    (path for path in [START, *START.parents] if (path / "configs/default.yaml").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("请从项目仓库内部运行此 Notebook。")

snapshot = json.loads(
    (ROOT / "reports/results_snapshot.json").read_text(encoding="utf-8")
)
print(f"仓库：{ROOT.name}")
print(f"结果快照生成时间： {snapshot['snapshot_generated_at_utc']}")

仓库：uk-road-collision-severity-prediction
结果快照生成时间： 2026-08-31T15:14:02.327952+00:00


## 1. 数据溯源

下方源文件哈希、处理契约和记录数来自受版本控制的结果快照。重新处理数据或训练模型后，
应重新生成该快照。

In [2]:
data = snapshot["data"]
display(pd.DataFrame(
    {
        "结果": [
            data["source_file"],
            data["source_sha256"],
            data["rows"],
            " to ".join(data["date_range"]),
            f"{data['ksi_share']:.2%}",
            data["contract_version"],
        ]
    },
    index=[
        "源文件",
        "源文件 SHA-256",
        "处理后记录数",
        "日期范围",
        "KSI 占比",
        "数据契约",
    ],
))

,结果
源文件,data/raw/dft-road-casualty-statistics-collisio...
源文件 SHA-256,8ce3f1290ea4830c041ddd737b543fb06b8667215208a1...
处理后记录数,513801
日期范围,2021-01-01 to 2025-12-31
KSI 占比,24.21%
数据契约,dft_open_dataset_2025


## 2. 描述性分析

以下图表已纳入版本控制，因此无需执行 Notebook 也能正常查看。

![严重程度构成](../reports/figures/processed/01_severity_composition.png)

![月度碰撞量与 KSI 趋势](../reports/figures/processed/20_monthly_time_series.png)

![照明条件](../reports/figures/processed/06_ksi_by_light.png)

![空间碰撞密度与严重程度](../reports/figures/processed/19_spatial_hex_analysis.png)

## 3. 五模型时间外验证

所有候选模型都使用 2021—2023 年训练，并在 2024 年进行模型比较。模型选择以
Average Precision 为主指标；选择过程中不使用 2025 年数据。

In [3]:
comparison = pd.DataFrame(snapshot["validation"]["models"])
comparison["model"] = comparison["model"].str.replace("_", " ").str.title()
comparison = comparison.rename(columns={
    "rank": "排名",
    "model": "模型",
    "roc_auc": "ROC-AUC",
    "average_precision": "平均精确率",
    "brier_score": "Brier 分数",
    "training_seconds": "训练时间（秒）",
})
display(comparison.round(4))

,排名,模型,ROC-AUC,平均精确率,Brier 分数,训练时间（秒）
0,1,Lightgbm,0.6558,0.3798,0.2311,5.9374
1,2,Catboost,0.6557,0.3792,0.2296,90.0490
2,3,Extra Trees,0.6443,0.3670,0.2242,28.5999
3,4,Logistic Regression,0.5894,0.3244,0.2431,7.0069
4,5,Dummy,0.5000,0.2484,0.1869,2.5112


![验证集模型对照](../reports/figures/model/model_validation_comparison.png)

![时间顺序学习曲线](../reports/figures/model/temporal_learning_curve.png)

## 4. 2025 年独立测试

分类阈值只在 2024 年验证集上通过最大化 KSI F1 确定，然后一次性应用于 2025 年测试集。
模型概率在按字面解释为风险前仍需校准。

In [4]:
test = snapshot["test"]
metrics = [
    ("入选模型", test["selected_model"]),
    ("分类阈值", test["threshold"]),
    ("ROC-AUC", test["roc_auc"]),
    ("平均精确率", test["average_precision"]),
    ("Brier 分数", test["brier_score"]),
    ("平衡准确率", test["balanced_accuracy"]),
    ("KSI 精确率", test["ksi_precision"]),
    ("KSI 召回率", test["ksi_recall"]),
    ("KSI F1", test["ksi_f1"]),
]
display(pd.DataFrame(metrics, columns=["指标", "结果"]).round(4))

,指标,结果
0,入选模型,lightgbm
1,分类阈值,0.46
2,ROC-AUC,0.639312
3,平均精确率,0.381905
4,Brier 分数,0.234938
5,平衡准确率,0.594959
6,KSI 精确率,0.328042
7,KSI 召回率,0.70042
8,KSI F1,0.446817


![排列重要性](../reports/figures/model/permutation_importance.png)

![精确率—召回率曲线](../reports/figures/model/precision_recall_curve.png)

![混淆矩阵](../reports/figures/model/confusion_matrix.png)

![校准曲线](../reports/figures/model/calibration_curve.png)

## 复现完整流程

在仓库根目录依次运行：

    python scripts/download_data.py
    python scripts/01_raw_analysis_and_processing.py
    python scripts/02_processed_analysis_and_visualisation.py
    python scripts/tune_lightgbm.py
    python scripts/03_model_training_and_visualisation.py
    python scripts/04_additional_visual_analysis.py
    python scripts/build_results_snapshot.py
    python -m pytest -q

环境配置、解读边界和输出路径见 [中文 README](../README.zh-CN.md)，详细结果见
[中文图表报告](../reports/figure_story_zh-CN.md)。